In [5]:
import sys, json

sys.path.append("used-car-price-predictor")

import numpy as np
import pandas as pd
from src.features import add_features, brand_and_model, NUM_FEATURES, CAT_FEATURES

df = pd.read_csv("used-car-price-predictor/data/processed/cars_clean.csv")
REF_YEAR = int(df["year"].max())
df[["brand", "model"]] = df["name"].apply(lambda n: pd.Series(brand_and_model(n)))
df = add_features(df, REF_YEAR)

json.dump({"ref_year": REF_YEAR}, open("used-car-price-predictor/models/meta.json", "w"))
print("Reference year:", REF_YEAR)
print("Brands:", df["brand"].nunique(), "| Models:", df["model"].nunique())
df[NUM_FEATURES + CAT_FEATURES].head()

Reference year: 2020
Brands: 32 | Models: 209


,car_age,km_driven,km_per_year,engine_cc,max_power_bhp,power_per_litre,mileage_kmpl,torque_nm,seats,owner_rank,brand,model,fuel,transmission,seller_type
0,6,145500,24250.000000,1248.0,74.00,59.294872,23.40,190.000000,5.0,1,Maruti,Swift,Diesel,Manual,Individual
1,6,120000,20000.000000,1498.0,103.52,69.105474,21.14,250.000000,5.0,2,Skoda,Rapid,Diesel,Manual,Individual
2,14,140000,10000.000000,1497.0,78.00,52.104208,17.70,124.544455,5.0,3,Honda,City,Petrol,Manual,Individual
3,10,127000,12700.000000,1396.0,90.00,64.469914,23.00,219.668960,5.0,1,Hyundai,i20,Diesel,Manual,Individual
4,13,120000,9230.769231,1298.0,88.20,67.950693,16.10,112.776475,5.0,1,Maruti,Swift,Petrol,Manual,Individual


In [5]:
import sys, os
sys.path.append("used-car-price-predictor")
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GroupKFold, cross_validate
from xgboost import XGBRegressor
from src.features import FEATURES, make_model

train = pd.read_csv("used-car-price-predictor/data/processed/train.csv")
X_train, y_train, g_train = train[FEATURES], train["selling_price"], train["group_id"]
cv = GroupKFold(n_splits=5)
os.makedirs("used-car-price-predictor/reports/model_results", exist_ok=True)

In [6]:
models = {
    "Median baseline":   DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Decision Tree":     DecisionTreeRegressor(max_depth=8, random_state=42),
    "Random Forest":     RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost":           XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                                      subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1),
}

In [7]:
scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error",
           "medae": "neg_median_absolute_error", "mape": "neg_mean_absolute_percentage_error", "r2": "r2"}

rows = []
for name, reg in models.items():
    res = cross_validate(make_model(reg), X_train, y_train, groups=g_train, cv=cv,
                         scoring=scoring, return_train_score=True)
    rows.append({
        "model": name,
        "MAE (lakh)":    -res["test_mae"].mean() / 1e5,
        "RMSE (lakh)":   -res["test_rmse"].mean() / 1e5,
        "MedAE (lakh)":  -res["test_medae"].mean() / 1e5,
        "MAPE %":        -res["test_mape"].mean() * 100,
        "R2 (CV)":        res["test_r2"].mean(),
        "R2 (train)":     res["train_r2"].mean(),
    })
    print("done:", name)

done: Median baseline
done: Linear Regression
done: Decision Tree
done: Random Forest
done: Gradient Boosting
done: XGBoost


In [9]:
results = pd.DataFrame(rows).sort_values("MAE (lakh)").reset_index(drop=True)
results["Overfit gap (R2)"] = results["R2 (train)"] - results["R2 (CV)"]
results = results.round(3)
results.to_csv("used-car-price-predictor/reports/model_results/baseline_comparison.csv", index=False)
results

,model,MAE (lakh),RMSE (lakh),MedAE (lakh),MAPE %,R2 (CV),R2 (train),Overfit gap (R2)
0,XGBoost,0.724,1.628,0.416,15.597,0.892,0.977,0.085
1,Random Forest,0.760,1.618,0.444,16.734,0.893,0.978,0.084
2,Linear Regression,0.790,1.600,0.452,16.795,0.894,0.900,0.005
3,Gradient Boosting,0.803,1.687,0.471,17.284,0.885,0.914,0.030
4,Decision Tree,0.942,1.923,0.547,20.657,0.844,0.944,0.100
5,Median baseline,2.746,5.087,1.900,79.819,-0.051,-0.050,0.001
